# Third candidate strain evaluation - *P.pentosaceus* and *S.thermophilus*

**Project:** Cooperative Tradeoff Analysis of Consortia for Plant-Based Fermentation 

## 1. Setup

In [1]:
# Setup 
import os, sys, cobra, warnings, re
warnings.filterwarnings('ignore', category=UserWarning, module='memote')

_lic = os.path.expanduser('~/gurobi.lic')
if os.path.exists(_lic):
    os.environ['GRB_LICENSE_FILE'] = _lic
cobra.Configuration().solver = 'gurobi'

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import gem_summary

## 2. *Pediococcus pentosaceus* ATCC 25745

The *P. pentosaceus* model was downloaded directly from the AGORA2 repository. *P. pentosaceus* was evaluated as a candidate LAB strain given its documented use in legume fermentation and reported capacity for BCAA catabolism via the Ehrlich pathway.

### 2.1 Model's Quality Assessment & Preprocessing

The AGORA2 SBML file requires preprocessing before loading with COBRApy: it contains a `groups` extension block (`listOfGroups`) that COBRApy's SBML parser does not support and which causes the import to fail. The fix below strips this block and its associated namespace declarations without altering any reaction, metabolite, or gene content.

In [2]:
def fix_agora_sbml(input_path, output_path):
    with open(input_path, 'r') as f:
        content = f.read()
    content = re.sub(
        r'<groups:listOfGroups>.*?</groups:listOfGroups>',
        '', content, flags=re.DOTALL
    )
    content = re.sub(r'\s*xmlns:groups="[^"]*"', '', content)
    content = re.sub(r'\s*groups:required="[^"]*"', '', content)
    with open(output_path, 'w') as f:
        f.write(content)

fix_agora_sbml('../models/raw/PpentosaceusATCC_25745.xml', '../models/raw/Pp_fixed.xml')
#quality analysis of iLP728
model_pp = cobra.io.read_sbml_model('../models/raw/Pp_fixed.xml')
results_pp = gem_summary(model_pp, label='Ppentosaceus')

Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpob0fnl42.lp
Reading time = 0.00 seconds
: 951 rows, 2125 columns, 9151 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp40_01sfw.lp
Reading time = 0.00 seconds
: 951 rows, 2125 columns, 9151 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         o

### 2.2 Legume Medium Growth Test

All exchange reactions are closed, then the canonical legume-medium composition is applied, using the same compound set and bounds as for *L.plantarum* and *L.mesenteroides*.

In [3]:
# Close all exchanges and apply legume-like medium
# using the same compound set as the canonical legume medium
with model_pp:
    for rxn in model_pp.exchanges:
        rxn.lower_bound = 0.0

    legume_like = {
        'EX_h2o(e)': -1000, 'EX_h(e)':     -1000,
        'EX_nh4(e)': -10,   'EX_so4(e)':   -10,
        'EX_pi(e)':  -10,   'EX_co2(e)':    0.0,
        'EX_glc(e)': -0.5,  'EX_sucr(e)':  -0.8,
        'EX_fru(e)': -0.1,
        'EX_glu_L(e)': -2.6, 'EX_asp_L(e)': -0.6,
        'EX_ala_L(e)': -0.3, 'EX_ser_L(e)': -0.3,
        'EX_leu_L(e)': -0.2, 'EX_val_L(e)': -0.1,
        'EX_ile_L(e)': -0.1, 'EX_phe_L(e)': -0.1,
        'EX_his_L(e)': -0.1, 'EX_lys_L(e)': -0.05,
        'EX_met_L(e)': -0.02,'EX_trp_L(e)': -0.02,
        'EX_thr_L(e)': -0.06,'EX_arg_L(e)': -0.1,
        'EX_ade(e)':   -1.0, 'EX_gua(e)':   -1.0,
        'EX_ura(e)':   -1.0, 'EX_thymd(e)': -1.0,
        'EX_nac(e)':   -0.01,'EX_pnto_R(e)':-0.01,
    }
    for ex_id, lb in legume_like.items():
        if ex_id in {r.id for r in model_pp.exchanges}:
         model_pp.reactions.get_by_id(ex_id).lower_bound = lb

    sol_leg = model_pp.optimize()
    print(f"Pp legume medium: μ = {sol_leg.objective_value:.4f} h⁻¹  [{sol_leg.status}]")

Pp legume medium: μ = 0.0000 h⁻¹  [optimal]


### 2.3 Root Cause Diagnosis - dipeptide dependency

Growth on the legume medium is zero. To identify why, the model is re-optimized with all exchanges unconstrained and the exchanges carrying active uptake flux are compared against the legume medium's compound set — any uptake the model relies on but that the legume medium does not supply is a candidate explanation for the zero-growth result.

In [4]:
# Identify which exchanges carry flux in unconstrained solution but are absent from the legume medium composition
with model_pp:
    for rxn in model_pp.exchanges:
        rxn.lower_bound = -1000.0
    sol_rich = model_pp.optimize()

    active_rich = {
        r.id: sol_rich.fluxes.get(r.id, 0)
        for r in model_pp.exchanges
        if sol_rich.fluxes.get(r.id, 0) < -1e-6
    }

legume_ids = set(legume_like.keys())
missing = {
    rid: flux for rid, flux in active_rich.items()
    if rid not in legume_ids
}

# Separate dipeptides from other missing compounds
# Excludes free amino acid IDs (ending in '_L'), which share the same residue-code substrings as dipeptides but are not dipeptides
dipeptides = {rid: f for rid, f in missing.items()
              if any(aa in rid for aa in ['ala', 'gly', 'leu', 'gln', 'thr', 'his', 'tyr']
              ) and len(rid.replace('EX_','').replace('(e)','')) <= 10
              and not rid.replace('EX_','').replace('(e)','').endswith('_L')}
other = {rid: f for rid, f in missing.items() if rid not in dipeptides}

print("Missing from legume medium — dipeptides:")
for rid, flux in sorted(dipeptides.items(), key=lambda x: x[1])[:15]:
    print(f"  {rid}: {flux:.4f}")

print("\nMissing from legume medium — other:")
for rid, flux in sorted(other.items(), key=lambda x: x[1])[:10]:
    print(f"  {rid}: {flux:.4f}")

print("\n")
print(dipeptides)
print(len(dipeptides))

Missing from legume medium — dipeptides:
  EX_glyleu(e): -1000.0000
  EX_alagln(e): -185.5811
  EX_alaglu(e): -22.6484
  EX_glypro(e): -6.7296
  EX_glycys(e): -3.0438
  EX_cgly(e): -0.3318

Missing from legume medium — other:
  EX_actn_R(e): -540.6360
  EX_gln_L(e): -322.8514
  EX_asn_L(e): -6.1988
  EX_tyr_L(e): -4.6405
  EX_hxan(e): -3.4284
  EX_uri(e): -2.6909
  EX_fol(e): -0.9954
  EX_dad_2(e): -0.7036
  EX_fe3(e): -0.6636
  EX_ribflv(e): -0.6636


{'EX_alagln(e)': np.float64(-185.58113457264218), 'EX_alaglu(e)': np.float64(-22.648360350940976), 'EX_cgly(e)': np.float64(-0.331807180209142), 'EX_glycys(e)': np.float64(-3.0437730744290747), 'EX_glyleu(e)': np.float64(-1000.0), 'EX_glypro(e)': np.float64(-6.72958804855072)}
6


## 3. Interpretation - *P. pentosaceus* exclusion

The AGORA2 model loads and grows in unconstrained conditions (μ = 41.79 h⁻¹), but this value is biologically unrealistic: 133
exchanges are open by default and the model has no nutritional constraints. Under the legume medium, growth is zero on both matrices.

The diagnosis reveals the cause: in the unconstrained solution, the model relies heavily on dipeptides as nitrogen and carbon
sources — `EX_alagln(e)`, `EX_alaglu(e)`, `EX_glyleu(e)`, `EX_glypro(e)`, `EX_glycys(e)`, `EX_cgly(e)` — at fluxes far exceeding those of individual amino acids (`EX_glyleu(e)` alone reaches the unconstrained bound of -1000 mmol/gDW/h). These dipeptides are not present in the legume medium composition, which is based on free amino acids and total protein hydrolysate fractions.

Restoring growth would require either redefining the medium to include peptide fractions — which lacks experimental support for chickpea and fava bean slurries — or rebuilding the nitrogen assimilation pathways in the model to function on free amino acids. Either path represents substantial model engineering that falls outside the scope of this project. Deep curation to make a
model compatible with a matrix it was never intended for is not the goal — the point is to use reconstructions that reflect the actual
metabolic capacity of the strain in this nutritional context.

For that reason, *P. pentosaceus* is excluded from further analysis.

## 4. *S.thermophilus* LMG 18311

The *S. thermophilus* model (MODEL1507180063) was retrieved from BioModels. *S.thermophilus* was evaluated as a candidate given its documented use as a starter culture in fermented plant-based products and its known Ehrlich pathway activity. The SBML file uses `_LPAREN_e_RPAREN_` encoding for compartment suffixes, which requires preprocessing before loading with COBRApy.

### 4.1 Model's Quality Assessment & Preprocessing

The fix below resolves the `_LPAREN_e_RPAREN_` encoding affecting reaction IDs. Metabolite IDs use a separate encoding (`_LSQBKTe_RSQBKT_`) that was not corrected, since it does not affect exchange reaction identification or the conclusions drawn in this notebook and the model is excluded from further use.

In [5]:
# St SBML uses URL-encoded parentheses for compartment suffixes
# _LPAREN_e_RPAREN_ → (e)
def fix_st_sbml(input_path, output_path):
    with open(input_path, 'r') as f:
        content = f.read()
    content = content.replace('_LPAREN_e_RPAREN_', '(e)')
    content = content.replace('_LPAREN_c_RPAREN_', '(c)')
    with open(output_path, 'w') as f:
        f.write(content)

fix_st_sbml('../models/raw/MODEL1507180063_url.xml', '../models/raw/St_fixed.xml')
m_st = cobra.io.read_sbml_model('../models/raw/St_fixed.xml')
results_st = gem_summary(m_st, label='St_MODEL1507180063')

'R_EX_ac(e)' is not a valid SBML 'SId'.
'R_EX_ala_L(e)' is not a valid SBML 'SId'.
'R_EX_arg_L(e)' is not a valid SBML 'SId'.
'R_EX_asp_L(e)' is not a valid SBML 'SId'.
'R_EX_cit(e)' is not a valid SBML 'SId'.
'R_EX_co2(e)' is not a valid SBML 'SId'.
'R_EX_cys_L(e)' is not a valid SBML 'SId'.
'R_EX_dha(e)' is not a valid SBML 'SId'.
'R_EX_diact(e)' is not a valid SBML 'SId'.
'R_EX_etoh(e)' is not a valid SBML 'SId'.
'R_EX_for(e)' is not a valid SBML 'SId'.
'R_EX_gal(e)' is not a valid SBML 'SId'.
'R_EX_gcald(e)' is not a valid SBML 'SId'.
'R_EX_gln_L(e)' is not a valid SBML 'SId'.
'R_EX_glu_L(e)' is not a valid SBML 'SId'.
'R_EX_glyc(e)' is not a valid SBML 'SId'.
'R_EX_h(e)' is not a valid SBML 'SId'.
'R_EX_h2o(e)' is not a valid SBML 'SId'.
'R_EX_his_L(e)' is not a valid SBML 'SId'.
'R_EX_ile_L(e)' is not a valid SBML 'SId'.
'R_EX_lac_L(e)' is not a valid SBML 'SId'.
'R_EX_lcts(e)' is not a valid SBML 'SId'.
'R_EX_leu_L(e)' is not a valid SBML 'SId'.
'R_EX_lys_L(e)' is not a valid SB

Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmpo501ylxy.lp
Reading time = 0.00 seconds
: 550 rows, 1113 columns, 5051 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q80000gn/T/tmp_4gra67m.lp
Reading time = 0.00 seconds
: 550 rows, 1113 columns, 5051 nonzeros
Set parameter Username
Set parameter LicenseID to value 2818537
Academic license - for non-commercial use only - expires 2027-05-05
         of a constant term in the left-hand side of a constraint.

Read LP format model from file /var/folders/pc/ykhcdwt55sv66yd1qmdsb5q8000

### 4.2 Legume Medium Growth Test

The legume medium composition is adapted to *S. thermophilus*'s exchange ID format. Chickpea and fava bean are tested separately, differing in galactose availability, to test whether the strain's reported preference for galactose over glucose is reflected in growth on each matrix.

In [6]:
# Apply legume medium using St's (e) compartment format
mapa_st = {
    'EX_glc(e)':    -0.5,  'EX_fru(e)':    -0.1,
    'EX_gal(e)':    -0.5,  'EX_sucr(e)':   -0.8,
    'EX_glu_L(e)':  -2.6,  'EX_asp_L(e)':  -0.6,
    'EX_ala_L(e)':  -0.3,  'EX_ser_L(e)':  -0.3,
    'EX_leu_L(e)':  -0.2,  'EX_val_L(e)':  -0.1,
    'EX_ile_L(e)':  -0.1,  'EX_phe_L(e)':  -0.1,
    'EX_his_L(e)':  -0.1,  'EX_lys_L(e)':  -0.05,
    'EX_met_L(e)':  -0.02, 'EX_cys_L(e)':  -0.02,
    'EX_trp_L(e)':  -0.02, 'EX_thr_L(e)':  -0.06,
    'EX_arg_L(e)':  -0.1,  'EX_pro_L(e)':  -0.03,
    'EX_tyr_L(e)':  -0.02, 'EX_gln_L(e)':  -0.1,
    'EX_nac(e)':    -0.01, 'EX_pnto_R(e)': -0.01,
    'EX_ribflv(e)': -0.01, 'EX_thm(e)':    -0.01,
    'EX_pi(e)':     -10,   'EX_h2o(e)':    -1000,
    'EX_h(e)':      -1000, 'EX_o2(e)':     -1.0,
}

with m_st:
    for rxn in m_st.exchanges:
        rxn.lower_bound = 0.0
    for ex_id, lb in mapa_st.items():
        try:
            m_st.reactions.get_by_id(ex_id).lower_bound = lb
        except KeyError:
            pass

    sol_cp = m_st.optimize()
    print(f"St chickpea:   μ = {sol_cp.objective_value:.4f} h⁻¹  [{sol_cp.status}]")

    # Fava bean — remove galactose (absent in significant quantities)
    m_st.reactions.get_by_id('EX_gal(e)').lower_bound = 0.0
    sol_fb = m_st.optimize()
    print(f"St fava bean:  μ = {sol_fb.objective_value:.4f} h⁻¹  [{sol_fb.status}]")

Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


St chickpea:   μ = 0.0347 h⁻¹  [optimal]
St fava bean:  μ = 0.0000 h⁻¹  [optimal]


### 4.3 Ehrlich pathway & Cross-Feeding Audit

Beyond the growth result, *S. thermophilus*'s relevance to the consortium depends on whether it can contribute Ehrlich pathway products (BCAA-derived aldehydes, alcohols, and acids) to the extracellular environment and whether it shares metabolite exchange capacity with *L.plantarum* and *L.mesenteroides*. The check below inspects internal pathway reactions, product exchange reactions and overlap with metabolites already tracked in the *L.plantarum*/*L.mesenteroides* models.

In [7]:
st_rxn_ids = {r.id for r in m_st.reactions}
st_ex_ids  = {r.id for r in m_st.exchanges}

# Ehrlich pathway reactions
print("Ehrlich pathway — internal reactions:")
ehrlich_rxns = {
    'BCAA transaminase (Leu)':   'LEUTA',
    'BCAA transaminase (Ile)':   'ILETA',
    'BCAA transaminase (Val)':   'VALTA',
    '2-ketoacid decarboxylase':  'KDC',
    'Alcohol dehydrogenase':     'ALCD2x',
}
for name, rxn_id in ehrlich_rxns.items():
    status = '✓' if rxn_id in st_rxn_ids else '✗ ABSENT'
    print(f"  {name:<35} {rxn_id:<10} {status}")

# Ehrlich product exchanges
print("\nEhrlich product exchanges:")
ehrlich_ex = {
    '3-methylbutanoate':  'EX_3mba(e)',
    '2-methylbutanoate':  'EX_2mba(e)',
    '2-methylpropanoate': 'EX_2mpa(e)',
    '3-methylbutanal':    'EX_3mbal(e)',
    '2-methylbutanal':    'EX_2mbal(e)',
    '3-methylbutanol':    'EX_iamoh(e)',
}
for name, ex_id in ehrlich_ex.items():
    status = '✓' if ex_id in st_ex_ids else '✗ ABSENT'
    print(f"  {name:<25} {ex_id:<20} {status}")

# Cross-feeding metabolite overlap with Lp/Lm
print("\nCross-feeding metabolite coverage:")
cross_mets = {
    'lactate':    'EX_lac_L(e)',
    'acetate':    'EX_ac(e)',
    'ethanol':    'EX_etoh(e)',
    'succinate':  'EX_succ(e)',
    'galactose':  'EX_gal(e)',
    'proline':    'EX_pro_L(e)',
    'GABA':       'EX_4abut(e)',
    'NH4':        'EX_nh4(e)',
    'adenine':    'EX_ade(e)',
}
for name, ex_id in cross_mets.items():
    status = '✓' if ex_id in st_ex_ids else '✗ ABSENT'
    print(f"  {name:<15} {ex_id:<20} {status}")

Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


Ehrlich pathway — internal reactions:
  BCAA transaminase (Leu)             LEUTA      ✓
  BCAA transaminase (Ile)             ILETA      ✓
  BCAA transaminase (Val)             VALTA      ✓
  2-ketoacid decarboxylase            KDC        ✗ ABSENT
  Alcohol dehydrogenase               ALCD2x     ✓

Ehrlich product exchanges:
  3-methylbutanoate         EX_3mba(e)           ✗ ABSENT
  2-methylbutanoate         EX_2mba(e)           ✗ ABSENT
  2-methylpropanoate        EX_2mpa(e)           ✗ ABSENT
  3-methylbutanal           EX_3mbal(e)          ✗ ABSENT
  2-methylbutanal           EX_2mbal(e)          ✗ ABSENT
  3-methylbutanol           EX_iamoh(e)          ✗ ABSENT

Cross-feeding metabolite coverage:
  lactate         EX_lac_L(e)          ✓
  acetate         EX_ac(e)             ✓
  ethanol         EX_etoh(e)           ✓
  succinate       EX_succ(e)           ✓
  galactose       EX_gal(e)            ✓
  proline         EX_pro_L(e)          ✓
  GABA            EX_4abut(e)          ✗ A

## 5. Interpretation - *S. thermophilus* exclusion

*S. thermophilus* grows on the chickpea medium at μ = 0.035 h⁻¹ but not on fava bean (μ = 0.000 h⁻¹). Inspecting active uptakes on chickpea revealed that galactose is the sole effective carbon source - glucose, sucrose and fructose carry no uptake flux. This reflects a genuine metabolic characteristic of *S. thermophilus*: the strain is specialized for lactose and galactose catabolism and lacks efficient glucose PTS transport. In fava bean, where galactose content is substantially lower, the model cannot sustain growth.

The Ehrlich pathway is incomplete at its decarboxylation step. LEUTA, ILETA and VALTA (BCAA transaminases) are present, but the model contains no 2-ketoacid decarboxylase (KDC) — the reactions initially checked, KARA1 and KARA2, are ketol-acid reductoisomerase isoforms belonging to BCAA biosynthesis, not the catabolic decarboxylation step required for Ehrlich-pathway aldehyde formation. Without this reaction, BCAA-derived 2-ketoacids cannot proceed to aldehydes regardless of exchange reaction availability. This means *S. thermophilus* cannot contribute to the predicted extracellular off-flavor profile through the Ehrlich pathway, independent of the growth limitation described above.

Restoring Ehrlich pathway function would require adding a KDC reaction de novo, not just exchange curation as applied to iLP728 and Koduru2022 - a different scale of model engineering, and one without direct genomic support in this reconstruction. Combined with growth restricted to a single carbon source at a very low rate and incompatibility with fava bean, *S. thermophilus* is unsuitable for the consortium modelling objectives defined here.

## 6. Bibliography

1. Heinken, A., Hertel, J., Acharya, G. et al. Genome-scale metabolic reconstruction of 7,302 human microorganisms for personalized medicine. Nat Biotechnol 41, 1320–1331 (2023). https://doi.org/10.1038/s41587-022-01628-0 

2. Pastink MI, Teusink B, Hols P, Visser S, de Vos WM, Hugenholtz J. Genome-scale model of Streptococcus thermophilus LMG18311 for metabolic comparison of lactic acid bacteria. Appl Environ Microbiol. 2009 Jun;75(11):3627-33. doi: 10.1128/AEM.00138-09. Epub 2009 Apr 3. PMID: 19346354; PMCID: PMC2687286.
